# AOGEC — Error Type Classification Baseline

In [ ]:
# 0) Install Dependencies
!pip -q install seaborn

In [ ]:
# 1) Import Libraries
import os, re, json, random, math
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import(
    AutoTokenizer, 
    T5EncoderModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import(
    classification_report, accuracy_score, f1_score, confusion_matrix
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 2) Reproduciblity
def set_seed(seed: int  =43) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# 3) Configuration
@dataclass
class CFG:
    # ── Model ──────────────────────────────────────────────────────
    model_name: str = "t5-base"
    device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

    # ── Data ───────────────────────────────────────────────────────
    data_dir: str = "/kaggle/input/datasets/solanadh/ao-gec-dataset-with-error-type"

    # ── Sequence lengths ───────────────────────────────────────────
    max_input_len: int = 128

    # ── Optimisation ──────────────────────────────────────────────
    batch_size:       int   = 8          
    grad_accum_steps: int   = 4
    
    lr_encoder:       float = 2e-5
    lr_cls_head:      float = 1e-4

    weight_decay:     float = 0.01
    warmup_ratio:     float = 0.06
    grad_clip:        float = 1.0
    num_workers:      int   = 2
    label_smoothing:  float = 0.05

    # ── Mixed precision ────────────────────────────────────────────
    amp_dtype: str = "bf16"

    # ── Gradient checkpointing ────────────────────────────────────
    grad_ckpt: bool = True

    # ── Training (single stage) ───────────────────────────────────
    num_epochs: int = 15          

    # ── Early stopping ─────────────────────────────────────────────
    patience: int = 5

    # ── Focal loss ────────────────────────────────────────────────
    focal_gamma: float = 1.5

    # ── Output ────────────────────────────────────────────────────
    output_dir: str = "/kaggle/working/aogec_cls_baseline_output"
    best_ckpt:  str = ""

    def __post_init__(self):
        self.train_path = f"{self.data_dir}/AOGEC_Dataset_Labeled_train.csv"
        self.val_path   = f"{self.data_dir}/AOGEC_Dataset_Labeled_eval.csv"
        self.test_path  = f"{self.data_dir}/AOGEC_Dataset_Labeled_test.csv"
        self.best_ckpt  = os.path.join(self.output_dir, "best_model.pt")
        os.makedirs(self.output_dir, exist_ok=True)
        print(f"Device : {self.device}")
        if self.device.type == "cuda":
            print(f"GPU    : {torch.cuda.get_device_name(0)}")
            if not torch.cuda.is_bf16_supported():
                self.amp_dtype = "fp16"
                print("GPU does not support bf16 → switched to fp16")
        else:
            self.amp_dtype = "none"
            self.grad_ckpt = False
            print("CPU mode — AMP and grad-ckpt disabled")
        print(f"AMP    : {self.amp_dtype}")

cfg = CFG()

_AMP_ENABLED = cfg.amp_dtype != "none"
_AMP_DTYPE   = torch.bfloat16 if cfg.amp_dtype == "bf16" else torch.float16
_autocast    = lambda: torch.amp.autocast(device_type="cuda", dtype=_AMP_DTYPE, enabled=_AMP_ENABLED)
scaler       = GradScaler(enabled=(_AMP_ENABLED and cfg.amp_dtype == "fp16"))

In [ ]:
# 4) Data Preprocessing & Label Encoding
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s([?.!,;:])", r"\1", text)
    return text.strip()

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    required = ["input", "target", "error_type"]
    df = df[required].copy()
    df.dropna(subset=required, inplace=True)
    for col in ["input", "target"]:
        df[col] = df[col].astype(str).apply(normalize_text)
    df["error_type"] = df["error_type"].astype(str).str.strip().str.lower()
        
    df.drop_duplicates(subset=required, inplace=True)
    return df.reset_index(drop=True)

train_df = clean_df(pd.read_csv(cfg.train_path, encoding="utf-8", encoding_errors="ignore"))
val_df   = clean_df(pd.read_csv(cfg.val_path,   encoding="utf-8", encoding_errors="ignore"))
test_df  = clean_df(pd.read_csv(cfg.test_path,  encoding="utf-8", encoding_errors="ignore"))

labels_sorted_train = sorted(train_df["error_type"].unique().tolist())
label2id = {lbl: i for i, lbl in enumerate(labels_sorted_train)}
id2label = {i: lbl for lbl, i in label2id.items()}


def map_labels(df: pd.DataFrame, mapping: dict, default: str, name: str = "") -> pd.DataFrame:
    unseen = set(df["error_type"]) - set(mapping)
    if unseen:
        print(f"{name}: unseen labels {unseen}")
    df = df.copy()
    df["error_id"] = df["error_type"].apply(lambda x: mapping.get(x, mapping.get(default, 0)))
    return df

default_label = labels_sorted_train[0]
train_df = map_labels(train_df, label2id, default_label, "train")
val_df   = map_labels(val_df,   label2id, default_label, "val")
test_df  = map_labels(test_df,  label2id, default_label, "test")

print(f"Data | Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")
print(f"Labels ({len(labels_sorted_train)}): {labels_sorted_train}")
print("\nTrain class distribution:")
print(train_df["error_type"].value_counts().to_string())

In [ ]:
# 5) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
_PAD_ID   = tokenizer.pad_token_id

In [ ]:
# 6) Prompt helper
def build_neutral_prompt(text: str) -> str:
    return f"classify the grammatical error in afan oromo: {text}"

In [ ]:
# 7) Dataset
class ClassificationDataset(Dataset):
    """
    Pre-tokenises the neutral prompt for each sample.
    Returns input_ids, attention_mask, and the integer error label.
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self._build()

    def _build(self):
        self.input_ids      = []
        self.attention_masks = []
        self.error_labels   = []

        for i in tqdm(range(len(self.df)), desc="Tokenising", leave=False):
            row      = self.df.iloc[i]
            prompt   = build_neutral_prompt(str(row["input"]))
            error_id = int(row["error_id"])

            enc = self.tokenizer(
                prompt,
                max_length=self.max_len,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            self.input_ids.append(enc["input_ids"].squeeze(0))
            self.attention_masks.append(enc["attention_mask"].squeeze(0))
            self.error_labels.append(torch.tensor(error_id, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "error_labels":   self.error_labels[idx],
            "raw_input":      row["input"],
            "raw_error":      row["error_type"],
        }


train_ds = ClassificationDataset(train_df, tokenizer, cfg.max_input_len)
val_ds   = ClassificationDataset(val_df,   tokenizer, cfg.max_input_len)
test_ds  = ClassificationDataset(test_df,  tokenizer, cfg.max_input_len)
print("Datasets ready.")

In [9]:
# 8) DataLoaders 
_loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device.type == "cuda"),
    persistent_workers=(cfg.num_workers > 0),
    prefetch_factor=(2 if cfg.num_workers > 0 else None),
)
train_loader = DataLoader(train_ds, shuffle=False,  drop_last=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)
print(f"DataLoaders ready | {len(train_loader)} train batches")

DataLoaders ready | 1584 train batches


In [ ]:
# 9) Model
class T5Classifier(nn.Module):
    """
    Encoder-only T5 classification model.

    Architecture
    ─────────────
    neutral_prompt → T5Encoder → mean+max pool → MLP → cls logits
    """

    def __init__(self, model_name: str, num_labels: int,
                 dropout: float = 0.1, grad_ckpt: bool = False):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        H = self.encoder.config.d_model

        if grad_ckpt:
            self.encoder.encoder.gradient_checkpointing_enable()
            print("Gradient checkpointing enabled on encoder.")

        self.dropout = nn.Dropout(dropout)

        # MLP
        self.classifier = nn.Sequential(
            nn.Linear(2 * H, H),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(H),
            
            nn.Linear(H, H // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            
            # NEW EXTENSION LAYER
            nn.Linear(H // 2, H // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(H // 4, num_labels),
        )

    @staticmethod
    def _pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Mean + max pooling over unmasked positions."""
        m      = mask.unsqueeze(-1).type_as(hidden)
        mean_p = (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)
        max_p  = hidden.masked_fill(m == 0, -1e9).max(1).values
        return torch.cat([mean_p, max_p], dim=-1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Returns cls_logits [B, num_labels].
        Single forward pass.
        """
        enc_out    = self.encoder(input_ids=input_ids,
                                  attention_mask=attention_mask,
                                  return_dict=True)
        pooled     = self._pool(enc_out.last_hidden_state, attention_mask)
        cls_logits = self.classifier(self.dropout(pooled))
        return cls_logits

    @torch.no_grad()
    def predict(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor) -> torch.Tensor:
        """Convenience wrapper — returns predicted class indices."""
        logits = self.forward(input_ids, attention_mask)
        return torch.argmax(logits.float(), dim=-1)

In [11]:
# 10) Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits  = logits.float()
        targets = targets.view(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = torch.exp(log_probs)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt     = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal  = -((1 - pt) ** self.gamma) * log_pt
        if self.weight is not None:
            focal = focal * self.weight.gather(0, targets)
        return focal.mean()

In [ ]:
# 11) Model + loss initialisation
num_labels    = len(label2id)
labels_sorted = [id2label[i] for i in range(num_labels)]

model = T5Classifier(
    cfg.model_name,
    num_labels=num_labels,
    grad_ckpt=cfg.grad_ckpt,
).to(cfg.device)

# Class-balanced focal loss weights
counts = (
    train_df.groupby("error_id").size()
    .reindex(range(num_labels), fill_value=0).values
)
cls_weights = torch.tensor(
    counts.max() / (counts + 1e-6), dtype=torch.float, device=cfg.device
)
cls_weights = cls_weights / cls_weights.mean()

focal_loss_fn = FocalLoss(gamma=cfg.focal_gamma, weight=cls_weights)
print(f"Model ready | params={sum(p.numel() for p in model.parameters()):,}")
print(f"Class weights: {dict(zip(labels_sorted, cls_weights.cpu().numpy().round(3)))}")

In [ ]:
# 12) Optimizer + Scheduler
def build_optimizer_and_scheduler(model, n_batches: int, n_epochs: int):
    enc_params = []
    cls_params = []
    seen = set()

    def _add(p, bucket):
        if id(p) not in seen and p.requires_grad:
            seen.add(id(p)); bucket.append(p)

    for name, param in model.named_parameters():
        if "classifier" in name:
            _add(param, cls_params)
        else:
            _add(param, enc_params)

    groups = [
        {"params": enc_params, "lr": cfg.lr_encoder},
        {"params": cls_params, "lr": cfg.lr_cls_head},
    ]
    groups = [g for g in groups if g["params"]]

    optimizer = AdamW(groups, weight_decay=cfg.weight_decay)

    effective_steps = math.ceil(n_batches / cfg.grad_accum_steps)
    total_steps     = effective_steps * n_epochs
    warmup_steps    = max(1, int(cfg.warmup_ratio * total_steps))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    print(f"  Optimizer | total_steps={total_steps} warmup={warmup_steps} "
          f"| LRs={[g['lr'] for g in groups]}")
    return optimizer, scheduler


optimizer, scheduler = build_optimizer_and_scheduler(
    model, len(train_loader), cfg.num_epochs
)

In [ ]:
# 13) Training loop
def train_one_epoch(
    model, loader, optimizer, scheduler, scaler
) -> float:
    """Returns mean classification loss for the epoch."""
    model.train()
    run_cls = 0.0
    optimizer.zero_grad()
    n_steps = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="  Train", leave=False)):
        ids   = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask  = batch["attention_mask"].to(cfg.device, non_blocking=True)
        elbl  = batch["error_labels"].to(cfg.device, non_blocking=True)

        with _autocast():
            logits   = model(ids, mask)             
            cls_loss = focal_loss_fn(logits, elbl)

        scaled = scaler.scale(cls_loss / cfg.grad_accum_steps)
        scaled.backward()
        run_cls += cls_loss.item()

        is_last = (step == n_steps - 1)
        if (step + 1) % cfg.grad_accum_steps == 0 or is_last:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    return run_cls / n_steps

In [ ]:
# 14) Validation loop
@torch.no_grad()
def validate(model, loader) -> Dict[str, Any]:
    model.eval()
    val_cls  = 0.0
    all_true, all_pred = [], []

    for batch in tqdm(loader, desc="  Val  ", leave=False):
        ids  = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask = batch["attention_mask"].to(cfg.device, non_blocking=True)
        elbl = batch["error_labels"].to(cfg.device, non_blocking=True)

        with _autocast():
            logits   = model(ids, mask)
            cls_loss = focal_loss_fn(logits, elbl)

        val_cls += cls_loss.item()
        preds = torch.argmax(logits.float(), dim=-1)
        all_true.extend(elbl.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

    return {
        "cls_loss":     val_cls / len(loader),
        "cls_acc":      accuracy_score(all_true, all_pred),
        "cls_f1_macro": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "true_cls":     all_true,
        "pred_cls":     all_pred,
    }

In [ ]:
# 15) Checkpoint helpers
def save_ckpt(model, epoch: int, f1: float):
    torch.save({
        "model_state_dict": model.state_dict(),
        "label_classes":    labels_sorted,
        "best_f1":          f1,
        "epoch":            epoch,
    }, cfg.best_ckpt)

In [ ]:
# 16) Training history + plots
history: List[Dict] = []

def save_plots():
    if not history:
        return
    df_h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("AOGEC Classification Baseline — Training History",
                 fontsize=13, fontweight="bold")

    axes[0].plot(df_h["epoch"], df_h["train_cls"], marker="o", label="Train")
    axes[0].plot(df_h["epoch"], df_h["val_cls_loss"], marker="s", label="Val")
    axes[0].set_title("Focal Classification Loss")
    axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(df_h["epoch"], df_h["val_cls_f1_macro"], marker="o", color="green")
    axes[1].set_title("Validation Macro-F1")
    axes[1].set_xlabel("Epoch")

    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "training_history.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Training plot saved → {path}")

In [ ]:
# 17) Single-stage training
best_f1          = -1.0
patience_counter =  0

print(f"\n{'='*60}")
print(f"Training classification baseline  ({cfg.num_epochs} epochs max)")
print(f"{'='*60}")

for epoch in range(1, cfg.num_epochs + 1):
    train_cls = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    val       = validate(model, val_loader)

    history.append({
        "epoch":            epoch,
        "train_cls":        train_cls,
        "val_cls_loss":     val["cls_loss"],
        "val_cls_f1_macro": val["cls_f1_macro"],
        "val_cls_acc":      val["cls_acc"],
    })

    print(f"[E{epoch:02d}/{cfg.num_epochs}] "
          f"train_cls={train_cls:.4f}  "
          f"val_cls={val['cls_loss']:.4f}  "
          f"f1={val['cls_f1_macro']:.4f}  "
          f"acc={val['cls_acc']:.4f}")

    if val["cls_f1_macro"] > best_f1:
        best_f1 = val["cls_f1_macro"]
        patience_counter = 0
        save_ckpt(model, epoch, best_f1)
        print(f"  → Checkpoint saved  (f1={best_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{cfg.patience})")
        if patience_counter >= cfg.patience:
            print("  → Early stopping."); break

# Save history JSON
history_path = os.path.join(cfg.output_dir, "training_history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

save_plots()
print(f"\nTraining complete. History → {history_path}")

In [ ]:
# 18) Final Test Evaluation
ckpt = torch.load(cfg.best_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model         = model.to(cfg.device).eval()
label_classes = ckpt["label_classes"]
label2id      = {l: i for i, l in enumerate(label_classes)}
id2label      = {i: l for l, i in label2id.items()}

print(f"\nBest model: Epoch {ckpt['epoch']} | F1={ckpt['best_f1']:.4f}")

all_true_cls, all_pred_cls = [], []
rows = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids      = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask     = batch["attention_mask"].to(cfg.device, non_blocking=True)
        true_ids = batch["error_labels"].cpu().tolist()

        with _autocast():
            preds = model.predict(ids, mask)

        for i in range(len(true_ids)):
            all_true_cls.append(true_ids[i])
            all_pred_cls.append(preds[i].item())
            rows.append([
                batch["raw_input"][i],
                batch["raw_error"][i],
                id2label[preds[i].item()],
            ])

# ── Metrics ───────────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("CLASSIFICATION PERFORMANCE (test set)")
print(classification_report(
    all_true_cls, all_pred_cls,
    target_names=label_classes, digits=4, zero_division=0,
))

# ── Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_true_cls, all_pred_cls)
n  = len(label_classes)
fig, ax = plt.subplots(figsize=(max(8, n), max(6, n - 1)))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_classes, yticklabels=label_classes, ax=ax,
)
ax.set_title("Afan Oromo Error Classification Baseline — Confusion Matrix")
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
plt.tight_layout()
cm_path = os.path.join(cfg.output_dir, "test_confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.show()
print(f"Confusion matrix saved → {cm_path}")

# ── Save predictions CSV ──────────────────────────────────────────────
rows_df  = pd.DataFrame(rows, columns=["input", "true_error", "pred_error"])
out_path = os.path.join(cfg.output_dir, "test_predictions.csv")
rows_df.to_csv(out_path, index=False)
print(f"Predictions CSV → {out_path}")

In [ ]:
# 19) Single-sentence inference
@torch.no_grad()
def predict_single(text: str) -> str:
    """Predict the error type for a single Afan Oromo sentence."""
    model.eval()
    enc = tokenizer(
        build_neutral_prompt(text),
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_input_len,
        padding="max_length",
    ).to(cfg.device)

    with _autocast():
        pred_id = model.predict(enc["input_ids"], enc["attention_mask"])

    return id2label.get(pred_id[0].item(), "unknown")


# ── Demo ──────────────────────────────────────────────────────────────
demo_samples = [
    "Abubakir fi Galgaloon bara kana barataa cimoo jedhamuu badhaafame ture .",
    "Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaa kabaje .",
    "Waan hundumaa darbuuf barbaachisa obsa .",
    "Ijaarani jiru isaan mana haaraa.",
    "Isheen kolfitee ture.",
]

print("\n" + "─" * 65)
print("DEMO — Afan Oromo Error Type Classifier (baseline)")
print("─" * 65)
for text in demo_samples:
    pred_error = predict_single(text)
    print(f"Input      : {text}")
    print(f"Error type : {pred_error}")
    print("─" * 65)